# 🩺 General Health Query Chatbot
### Task 4 — DevelopersHub AI/ML Engineering Internship
**Model:** OpenRouter API (free tier) · **Technique:** Prompt Engineering + Safety Filters


## Step 1 — Install Libraries

In [ ]:
!pip install requests -q
print("✅ Libraries ready")


✅ Libraries ready


## Step 2 — Import Libraries & Set API Key

> 🔑 Get your **free** API key from [openrouter.ai/keys](https://openrouter.ai/keys) — sign up takes 30 seconds, no credit card needed.

In [ ]:
import requests
import re
import textwrap
import time # Import the time module for sleep functionality
from datetime import datetime

# ─────────────────────────────────────────────────────────────────
#   PASTE YOUR OPENROUTER API KEY BELOW (keep the quotes)
# ─────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = "Enter your API Key here."

# Free model — no credits needed on OpenRouter
MODEL = "openai/gpt-3.5-turbo"

# Confirm
if "PASTE" in OPENROUTER_API_KEY:
    print("⚠️  Please replace OPENROUTER_API_KEY with your actual key above, then re-run this cell.")
else:
    print("✅ API key set.")
    print(f"   Model : {MODEL}")
    print(f"   Time  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")


✅ API key set.
   Model : openai/gpt-3.5-turbo
   Time  : 2026-05-14 09:17


## Step 3 — Test API Connection

Run this cell to confirm your API key works before building the chatbot.

In [ ]:
def call_openrouter(messages, max_tokens=400, max_retries=10, initial_delay=2):
    """
    Sends a list of messages to the OpenRouter API and returns
    the assistant's reply as a plain string, with exponential backoff for retries.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "HealthBot - DevelopersHub Internship"
    }
    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0.6
    }

    retry_count = 0
    current_delay = initial_delay

    while retry_count < max_retries:
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=30
            )

            if response.status_code == 429: # Rate limit exceeded
                print(f"⚠️  Rate limit exceeded. Retrying in {current_delay} seconds...")
                time.sleep(current_delay)
                current_delay *= 2 # Exponential backoff
                retry_count += 1
                continue # Try again

            if response.status_code != 200:
                raise Exception(f"API Error {response.status_code}: {response.text}")

            return response.json()["choices"][0]["message"]["content"].strip()

        except requests.exceptions.Timeout:
            print(f"⚠️  API request timed out. Retrying in {current_delay} seconds...")
            time.sleep(current_delay)
            current_delay *= 2
            retry_count += 1
        except Exception as e:
            # Catch other potential errors, but re-raise if not a retryable issue
            if "429" in str(e): # Sometimes 429 can be caught as a generic exception
                print(f"⚠️  Rate limit exceeded (generic exception). Retrying in {current_delay} seconds...")
                time.sleep(current_delay)
                current_delay *= 2
                retry_count += 1
                continue
            raise e # Re-raise if it's a different error

    raise Exception(f"Failed to get a response from OpenRouter API after {max_retries} retries.")


# ── Quick connection test ──────────────────────────────────────────────────────
print("Testing API connection...")
try:
    test_reply = call_openrouter([{"role": "user", "content": "Say hello in one sentence."}])
    print(f"✅ API connection successful!")
    print(f"   Model response: {test_reply}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   → Check your API key and try again.")


Testing API connection...
✅ API connection successful!
   Model response: Hello!


## Step 4 — Prompt Engineering Design

The **system prompt** is the core of this project. It tells the AI:
- Who it is (a friendly health assistant called HealthBot)
- What it must **never** do (diagnose, prescribe, give emergency advice without disclaimers)
- How to format every reply (simple language, 3–5 sentences, always suggest a doctor for serious issues)


In [ ]:
# ── System prompt — shapes every single response ──────────────────────────────
SYSTEM_PROMPT = """You are HealthBot, a friendly and knowledgeable health information assistant.

Your purpose is to provide general health information in plain, simple language
that anyone can understand. You are warm, supportive, and non-judgmental.

You MUST follow these rules in EVERY response without exception:
1. NEVER diagnose any medical condition.
2. NEVER prescribe or recommend specific medications or exact dosages.
3. NEVER give emergency instructions without telling the user to call emergency services first.
4. NEVER make definitive claims about what illness a specific person has.
5. ALWAYS recommend consulting a qualified doctor or pharmacist for serious,
   persistent, or worsening symptoms.
6. If a question is outside safe general health information, politely decline
   and redirect the user to a healthcare professional.

Format rules:
- Use plain English — explain any medical term you use.
- Keep answers to 3–5 sentences maximum.
- End with a brief, friendly reminder to see a doctor if symptoms persist.
- Tone: warm, clear, supportive — like a knowledgeable friend, not a textbook.
"""

print("✅ System prompt defined.")
print(f"   Length: {len(SYSTEM_PROMPT)} characters")
print()
print("Preview (first 200 chars):")
print(SYSTEM_PROMPT[:200] + "...")


✅ System prompt defined.
   Length: 1095 characters

Preview (first 200 chars):
You are HealthBot, a friendly and knowledgeable health information assistant.

Your purpose is to provide general health information in plain, simple language
that anyone can understand. You are warm,...


## Step 5 — Safety Filter Implementation

The safety filter is a **separate layer** that runs BEFORE the AI is called.
It catches dangerous patterns using regex and returns a pre-written safe
response — the model never even sees these queries.

**Five categories blocked:**
- 🚨 Emergency symptoms (chest pain, can't breathe, overdose, seizure)
- 💊 Prescription requests (prescribe me, exact dose, what medication should I take)
- 🔬 Diagnosis requests (do I have X, diagnose me)
- 💔 Self-harm queries
- ⚗️ Harmful substance requests


In [ ]:
# ── Emergency patterns ────────────────────────────────────────────────────────
EMERGENCY_PATTERNS = [
    r"\bchest\s*pain\b",
    r"\bcan'?t\s+breathe\b",
    r"\bdifficulty\s+breath\w*\b",
    r"\bsevere\s+(bleeding|pain|allergic)\b",
    r"\bheart\s+attack\b",
    r"\bstroke\b",
    r"\bunconscious\b",
    r"\bseizure\b",
    r"\boverdos\w+\b",
    r"\banaphylax\w+\b",
    r"\bsuicid\w+\b",
    r"\bkill\s+myself\b",
    r"\bself.?harm\b",
]

EMERGENCY_RESPONSE = """⚠️  This sounds like it could be a medical emergency.

Please call emergency services IMMEDIATELY:
  🇬🇧 UK       : 999 (ambulance) or 111 (NHS non-emergency)
  🇵🇰 Pakistan : 1122 (Rescue) · 115 (Edhi) · 1021 (Aman)
  🌍 Global    : 112

Do not wait — please seek help right now. 💙"""

# ── Prescription patterns ─────────────────────────────────────────────────────
PRESCRIPTION_PATTERNS = [
    r"\bprescribe\s+me\b",
    r"\bgive\s+me\s+a\s+prescription\b",
    r"\bhow\s+much\s+.{1,25}\s+should\s+i\s+take\b",
    r"\bexact\s+(dose|dosage)\b",
    r"\bwhat\s+medication\s+should\s+i\s+(take|use)\b",
]

PRESCRIPTION_RESPONSE = """I'm not able to recommend specific medications or dosages — that requires
a qualified doctor or pharmacist who knows your full medical history.

Please visit your GP, a pharmacist, or an urgent care clinic.
They will give you safe, personalised advice. 🏥"""

# ── Diagnosis patterns ────────────────────────────────────────────────────────
DIAGNOSIS_PATTERNS = [
    r"\bdo\s+i\s+have\b",
    r"\bdiagnose\s+me\b",
    r"\bam\s+i\s+(sick|ill|diabetic|pregnant|infected|positive)\b",
    r"\bwhat\s+(disease|condition|illness)\s+do\s+i\s+have\b",
    r"\bis\s+it\s+(cancer|diabetes|covid|hiv|tb)\b",
]

DIAGNOSIS_RESPONSE = """I can share general health information, but I'm not able to diagnose
any medical condition — only a qualified doctor can do that after a
proper in-person examination.

If you're worried about specific symptoms, please book an appointment
with your GP as soon as possible. Early check-ups are always a good idea. 🩺"""

# ── Harmful patterns ──────────────────────────────────────────────────────────
HARMFUL_PATTERNS = [
    r"\bhow\s+to\s+(make|synthesize|produce)\s+.{1,20}(drug|poison|narcotic)\b",
    r"\billegal\s+(drug|substance)\b",
]

HARMFUL_RESPONSE = """I'm not able to help with that request.
If you're struggling with substance-related concerns, please reach out to a
healthcare professional or a support helpline. You deserve proper care. 💙"""


def pre_filter(user_input):
    """
    Scans user input BEFORE sending to the LLM.
    Returns (is_blocked, override_response).
    If is_blocked is True, return override_response directly without calling the API.
    """
    text = user_input.lower()

    for pattern in EMERGENCY_PATTERNS:
        if re.search(pattern, text):
            return True, EMERGENCY_RESPONSE

    for pattern in PRESCRIPTION_PATTERNS:
        if re.search(pattern, text):
            return True, PRESCRIPTION_RESPONSE

    for pattern in DIAGNOSIS_PATTERNS:
        if re.search(pattern, text):
            return True, DIAGNOSIS_RESPONSE

    for pattern in HARMFUL_PATTERNS:
        if re.search(pattern, text):
            return True, HARMFUL_RESPONSE

    return False, ""


def post_filter(model_output):
    """
    Scans the model's response for any risky phrases that slipped through.
    Appends a safety disclaimer if found.
    """
    risky_phrases = [
        "you have ", "you are suffering from", "this is definitely",
        "i diagnose", "take exactly", "take x mg",
    ]
    lower = model_output.lower()
    for phrase in risky_phrases:
        if phrase in lower:
            model_output += (
                "\n\n⚠️  Note: This is general information only. "
                "Always consult a qualified healthcare professional for personal medical advice."
            )
            break
    return model_output


print("✅ Safety filters defined.")
print(f"   Emergency patterns : {len(EMERGENCY_PATTERNS)}")
print(f"   Prescription rules : {len(PRESCRIPTION_PATTERNS)}")
print(f"   Diagnosis rules    : {len(DIAGNOSIS_PATTERNS)}")
print(f"   Harmful patterns   : {len(HARMFUL_PATTERNS)}")


✅ Safety filters defined.
   Emergency patterns : 13
   Prescription rules : 5
   Diagnosis rules    : 5
   Harmful patterns   : 2


## Step 6 — Chatbot Query Handler

The `ask_healthbot()` function is the **brain** of the system. It runs the
full pipeline in order:

```
User Input → Pre-filter → Build messages → Call OpenRouter API → Post-filter → Response
```

`chat()` is the convenience wrapper that also prints the formatted output.


In [ ]:
# Keeps the full conversation in memory for the session
conversation_history = []


def ask_healthbot(user_query, remember_history=False):
    """
    Main chatbot function.

    Args:
        user_query       : The user's health question (string).
        remember_history : If True, keeps conversation context across calls.

    Returns:
        Safe, formatted response string.
    """
    if not user_query.strip():
        return "Please enter a health question and I'll do my best to help!"

    # ── Step 1: Pre-filter check ───────────────────────────────────────────────
    is_blocked, override = pre_filter(user_query)
    if is_blocked:
        return override

    # ── Step 2: Build message list for the API ────────────────────────────────
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if remember_history and conversation_history:
        messages.extend(conversation_history)

    messages.append({"role": "user", "content": user_query})

    # ── Step 3: Call the OpenRouter API ──────────────────────────────────────
    try:
        raw_output = call_openrouter(messages, max_tokens=400)
    except Exception as e:
        return f"⚠️  API error: {e}\nPlease check your API key and internet connection."

    # ── Step 4: Post-filter check ─────────────────────────────────────────────
    safe_output = post_filter(raw_output)

    # ── Step 5: Store in history (optional) ───────────────────────────────────
    if remember_history:
        conversation_history.append({"role": "user",      "content": user_query})
        conversation_history.append({"role": "assistant", "content": safe_output})

    return safe_output.strip()


def chat(query, remember_history=False):
    """Runs ask_healthbot() and prints a formatted response box."""
    divider = "─" * 62
    response = ask_healthbot(query, remember_history)

    print()
    print(divider)
    print(f"  👤 You      : {query}")
    print(divider)
    print(f"  🤖 HealthBot:")
    print()
    # Word-wrap the response for clean display
    for para in response.split("\n"):
        if para.strip():
            for line in textwrap.wrap(para, width=58):
                print(f"     {line}")
        else:
            print()
    print()
    print(f"  🕐 {datetime.now().strftime('%H:%M:%S')}")
    print(divider)


def reset_conversation():
    """Clears conversation history to start a fresh session."""
    global conversation_history
    conversation_history = []
    print("✅ Conversation history cleared.")


print("✅ Chatbot engine ready.")
print("   Usage: chat('your health question here')")


✅ Chatbot engine ready.
   Usage: chat('your health question here')


## Step 7 — Test Case A: General Health Questions

These are standard wellness queries. The full pipeline runs: pre-filter
(passes), prompt engineering applied, OpenRouter API called, post-filter
checked, response printed.


In [ ]:
print("=" * 62)
print("  TEST A — GENERAL HEALTH QUESTIONS")
print("=" * 62)

chat("What causes a sore throat?")
chat("What are the symptoms of dehydration?")
chat("How can I improve my sleep quality?")


  TEST A — GENERAL HEALTH QUESTIONS

──────────────────────────────────────────────────────────────
  👤 You      : What causes a sore throat?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     A sore throat can be caused by various factors, such as
     viral infections like the common cold or flu, bacterial
     infections like strep throat, irritants like smoke or
     pollution, allergies, dry air, or even shouting or singing
     loudly. Drinking plenty of fluids, resting your voice, and
     using lozenges can help soothe a sore throat. If your sore
     throat persists or is severe, it's best to see a doctor
     for proper evaluation and treatment. Take care of
     yourself!

  🕐 09:17:55
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
  👤 You      : What are the symptoms of dehydration?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

  

## Step 7B — Test Case B: Medication Questions (General Info)

These questions ask about medications in a general, safe way. HealthBot
provides general information without prescribing or giving specific dosages.


In [ ]:
print("=" * 62)
print("  TEST B — MEDICATION QUESTIONS (GENERAL INFO)")
print("=" * 62)

chat("Is paracetamol safe for children?")
chat("What are common side effects of antibiotics?")


  TEST B — MEDICATION QUESTIONS (GENERAL INFO)

──────────────────────────────────────────────────────────────
  👤 You      : Is paracetamol safe for children?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     Yes, paracetamol is generally safe for children when used
     correctly and at the appropriate dose. It can help reduce
     fever and relieve mild to moderate pain. However, it's
     important to follow the recommended dosage based on your
     child's age and weight to avoid any potential side
     effects. Always consult a pediatrician or pharmacist for
     the correct dosage for your child. If you have any
     concerns or notice any unusual symptoms, it's best to seek
     medical advice promptly.

     ⚠️  Note: This is general information only. Always consult
     a qualified healthcare professional for personal medical
     advice.

  🕐 09:17:59
──────────────────────────────────────────────────────────────

───────────────────────────

## Step 7C — Test Case C: Safety Filter Tests

These queries should **never** reach the API. The pre-filter catches them
and returns a safe, pre-written response immediately.


In [ ]:
print("=" * 62)
print("  TEST C — SAFETY FILTER TESTS (API never called)")
print("=" * 62)

chat("I have chest pain and can't breathe")     # → Emergency override
chat("Diagnose me — do I have diabetes?")        # → Diagnosis block
chat("Prescribe me something for my headache")   # → Prescription block


  TEST C — SAFETY FILTER TESTS (API never called)

──────────────────────────────────────────────────────────────
  👤 You      : I have chest pain and can't breathe
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     ⚠️  This sounds like it could be a medical emergency.

     Please call emergency services IMMEDIATELY:
       🇬🇧 UK       : 999 (ambulance) or 111 (NHS non-emergency)
       🇵🇰 Pakistan : 1122 (Rescue) · 115 (Edhi) · 1021 (Aman)
       🌍 Global    : 112

     Do not wait — please seek help right now. 💙

  🕐 09:18:00
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
  👤 You      : Diagnose me — do I have diabetes?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     I can share general health information, but I'm not able
     to diagnose
     any medical condition — only a qualified doctor can do
     that after a
     proper in-perso

## Step 7D — Test Case D: Multi-Turn Conversation

With `remember_history=True`, HealthBot remembers previous messages in the
same session — so follow-up questions make sense in context.


In [ ]:
print("=" * 62)
print("  TEST D — MULTI-TURN CONVERSATION")
print("=" * 62)

reset_conversation()

chat("What are the early signs of a cold?", remember_history=True)
chat("How long does it usually last?", remember_history=True)   # "it" = the cold
chat("What can help speed up recovery?", remember_history=True)


  TEST D — MULTI-TURN CONVERSATION
✅ Conversation history cleared.

──────────────────────────────────────────────────────────────
  👤 You      : What are the early signs of a cold?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     Early signs of a cold can include a runny or stuffy nose,
     sore throat, sneezing, coughing, and mild fatigue. You
     might also experience a low-grade fever or headache. These
     symptoms usually develop gradually. Remember, if you're
     feeling unwell, it's always a good idea to rest, stay
     hydrated, and consider seeing a healthcare professional if
     your symptoms persist or worsen.

  🕐 09:18:01
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
  👤 You      : How long does it usually last?
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     A common cold typically lasts about 7-10 days, but
     sym

## Step 8 — Interactive Chat Session

Run this cell to open a live chat where you type your own questions.
Type `quit` or `exit` to end the session.


In [ ]:
print("💬 HealthBot — Interactive Chat")
print("   Type 'quit' or 'exit' to end the session.")
print("   Type 'reset' to clear conversation history.")
print()

reset_conversation()

while True:
    try:
        user_input = input("You: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n💙 Session ended. Stay healthy!")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "bye"):
        print("HealthBot: Goodbye! Take care of yourself. 💙")
        break

    if user_input.lower() == "reset":
        reset_conversation()
        continue

    chat(user_input, remember_history=True)


💬 HealthBot — Interactive Chat
   Type 'quit' or 'exit' to end the session.
   Type 'reset' to clear conversation history.

✅ Conversation history cleared.
You: Hi 

──────────────────────────────────────────────────────────────
  👤 You      : Hi
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     Hello! How can I help you today with any health-related
     questions you might have?

  🕐 09:18:15
──────────────────────────────────────────────────────────────
You: I have sore throat 

──────────────────────────────────────────────────────────────
  👤 You      : I have sore throat
──────────────────────────────────────────────────────────────
  🤖 HealthBot:

     I'm sorry to hear that you have a sore throat. It could be
     due to various reasons like a viral infection, allergies,
     or even dry air. Gargling with warm salt water, staying
     hydrated, and resting can help soothe the discomfort. If
     your symptoms persist or worsen, please consider

## Step 9 — Evaluation & Scoring

We score each test case by checking how many expected keywords appear in
the response. This gives a rough but transparent quality measure.


In [ ]:
TEST_CASES = [
    ("What causes a sore throat?",               ["virus","bacteria","infection","irritation","throat"]),
    ("What are the symptoms of dehydration?",     ["thirst","urine","dizzy","dry","fatigue"]),
    ("Is paracetamol safe for children?",         ["children","weight","dose","pharmacist","doctor"]),
    ("How can I improve my sleep quality?",       ["sleep","routine","screen","caffeine","relax"]),
    ("What are common side effects of antibiotics?", ["nausea","diarrhoea","stomach","rash","complete"]),
    ("I have chest pain and can't breathe",       ["emergency","999","112","immediately"]),
    ("Diagnose me — do I have diabetes?",         ["diagnose","doctor","gp","examination"]),
    ("Prescribe me something for my headache",    ["prescribe","pharmacist","professional","gp"]),
]

print("Running evaluation — this calls the API for non-blocked queries...")
print()
print("=" * 68)
print(f"  {'#':<3} {'Query (truncated)':<35} {'Hits':<10} {'Score'}")
print("─" * 68)

total = 0
results = []

for i, (query, keywords) in enumerate(TEST_CASES, 1):
    response  = ask_healthbot(query)
    resp_lower = response.lower()
    hits      = [kw for kw in keywords if kw in resp_lower]
    score     = len(hits) / len(keywords)
    total    += score
    bar       = "█" * round(score * 10) + "░" * (10 - round(score * 10))
    label     = query[:33] + ".." if len(query) > 33 else query
    print(f"  {i:<3} {label:<35} {len(hits)}/{len(keywords):<8} {bar} {score*100:.0f}%")
    results.append((query, response, score))

avg = (total / len(TEST_CASES)) * 100
print("─" * 68)
print(f"  {'AVERAGE SCORE':<48} {avg:.1f}%")
print("=" * 68)


Running evaluation — this calls the API for non-blocked queries...

  #   Query (truncated)                   Hits       Score
────────────────────────────────────────────────────────────────────
  1   What causes a sore throat?          3/5        ██████░░░░ 60%
  2   What are the symptoms of dehydrat.. 4/5        ████████░░ 80%
  3   Is paracetamol safe for children?   5/5        ██████████ 100%
  4   How can I improve my sleep qualit.. 5/5        ██████████ 100%
  5   What are common side effects of a.. 3/5        ██████░░░░ 60%
  6   I have chest pain and can't breat.. 4/4        ██████████ 100%
  7   Diagnose me — do I have diabetes?   4/4        ██████████ 100%
  8   Prescribe me something for my hea.. 2/4        █████░░░░░ 50%
────────────────────────────────────────────────────────────────────
  AVERAGE SCORE                                    81.2%


## Step 10 — Strengths, Limitations & Final Insights


In [ ]:
analysis = {
    "✅  Strengths": [
        "Three-layer safety: pre-filter → prompt engineering → post-filter",
        "Emergency queries intercepted before the API is ever called",
        "Multi-turn memory — HealthBot remembers context within a session",
        "OpenRouter gives access to powerful free models (Llama 3.2, Mistral, etc.)",
        "Swap any model in one line without touching the safety logic",
        "Consistent professional tone enforced by the system prompt",
    ],
    "❌  Limitations": [
        "No persistent memory between separate Colab sessions",
        "Regex filters may miss cleverly rephrased dangerous queries",
        "Response quality depends on the free model tier used",
        "English only — no multilingual support in this version",
        "No real-time medical database or NHS guidelines integration",
    ],
    "🚀  Next Steps for Production": [
        "Add RAG pipeline connected to NHS/WHO/NICE guidelines",
        "Replace regex pre-filter with an LLM-based safety classifier",
        "Build a Streamlit or Gradio web interface for non-technical users",
        "Add user feedback (thumbs up/down) to improve prompts over time",
        "Log and audit all queries for ongoing clinical safety review",
    ],
}

for section, points in analysis.items():
    print(f"\n{section}")
    print("─" * 55)
    for p in points:
        print(f"  • {p}")

print()
print("─" * 62)
print("  ⚠️  DISCLAIMER")
print("─" * 62)
print("""  This chatbot is for EDUCATIONAL purposes only.
  It is NOT a medical device and must never replace
  professional medical advice, diagnosis, or treatment.
  Always consult a qualified healthcare professional.""")
print("─" * 62)



✅  Strengths
───────────────────────────────────────────────────────
  • Three-layer safety: pre-filter → prompt engineering → post-filter
  • Emergency queries intercepted before the API is ever called
  • Multi-turn memory — HealthBot remembers context within a session
  • OpenRouter gives access to powerful free models (Llama 3.2, Mistral, etc.)
  • Swap any model in one line without touching the safety logic
  • Consistent professional tone enforced by the system prompt

❌  Limitations
───────────────────────────────────────────────────────
  • No persistent memory between separate Colab sessions
  • Regex filters may miss cleverly rephrased dangerous queries
  • Response quality depends on the free model tier used
  • English only — no multilingual support in this version
  • No real-time medical database or NHS guidelines integration

🚀  Next Steps for Production
───────────────────────────────────────────────────────
  • Add RAG pipeline connected to NHS/WHO/NICE guidelines
  •